# Hybrid ANN–Proteus framework — Phase 1 (Python)Notebook entry point for the executed pipeline. Runs the same code as`src/train_ann.py`; nothing here is a reimplementation.**Honesty contract.** No FFT, THD or DWT is computed: the supplied CSVs hold onescalar sample per channel per row, so those window operations are undefined on them.The MATLAB, Proteus and ETAP stages are not executed here.Set `FAULT_DATA_DIR` to the folder holding `classData.csv` and `detect_dataset.csv`.

In [ ]:
import os, sysos.environ.setdefault("FAULT_DATA_DIR", "../archive")   # <- adjust if neededsys.path.insert(0, "src")import numpy as np, pandas as pdprint("data dir:", os.environ["FAULT_DATA_DIR"])

## 1. Audit the supplied files

In [ ]:
from train_ann import load_data, label_audit, iqr_screencd, dd = load_data()vec = label_audit(cd)iqr_screen(cd)vec.value_counts()

## 2. Verify the label encoding against measured currentsThe Blueprint states LL = `0011` (A–B). That vector does not occur. The data contains`0110`; the cell below shows it is an ungrounded **B–C** fault.

In [ ]:
I0 = (cd[['Ia','Ib','Ic']].sum(axis=1)/3)ev = cd.assign(vec=vec).groupby('vec').apply(    lambda g: pd.Series({        '|Ia|': g.Ia.abs().mean(), '|Ib|': g.Ib.abs().mean(), '|Ic|': g.Ic.abs().mean(),        '|I0|': (g[['Ia','Ib','Ic']].sum(axis=1)/3).abs().mean(), 'n': len(g)}),    include_groups=False)ev.round(3)

## 3. Build the 24 row-wise features

In [ ]:
from feature_engineering import build_rowwise_features, FEATURE_NAMESfrom train_ann import clip_1_99, MEASX = build_rowwise_features(clip_1_99(cd, MEAS))print(X.shape); X.head(3)

### The window features are guarded, not silently computed

In [ ]:
from feature_engineering import window_rms, thd, dwt_energytry:    window_rms([0.42])          # a single scalar sampleexcept ValueError as e:    print("correctly refused:", e)

## 4. Run the full pipelineTrains the exact Keras model across all four formulations plus the SMOTE diagnostic,and writes every artefact to `out/`. Roughly 3–4 minutes on CPU.

In [ ]:
from train_ann import mainmain()

## 5. Read back the measured metrics

In [ ]:
m = pd.read_csv("out/metrics.csv")m[(m.experiment=="A") & (m.task=="multi-label")][["metric","value","unit"]]

In [ ]:
m[m.task=="binary-detection"][["metric","value","unit"]]

## 6. Why classification caps near 85 %90.7 % of six-class errors are LLL↔LLLG confusions. The two classes are separated by a0.067 A band of zero-sequence current that min-max scaling compresses to ~1e-4 of thefeature axis.

In [ ]:
lll, lllg = I0[vec=='0111'].abs(), I0[vec=='1111'].abs()lo, hi = lll.min(), lll.max()print(f"LLL  band      : [{lo:.4f}, {hi:.4f}] A   width {hi-lo:.4f} A")print(f"LLLG inside    : {((lllg>=lo)&(lllg<=hi)).mean()*100:.2f} %")print(f"LLLG below     : {(lllg<lo).mean()*100:.2f} %")span = (cd[['Ia','Ib','Ic']].sum(axis=1)/3)print(f"band / full range of I_zero_seq : {(hi-lo)/(span.max()-span.min()):.3e}")

### The band is a simulation artefactWithin-class dispersion of the ungrounded classes is near zero, and every class occupiesa contiguous block of rows — one fixed-parameter simulation run each. A model tuned toexploit this band would not generalise.

In [ ]:
names={'0000':'No fault','1001':'LG','0110':'LL','1011':'LLG','0111':'LLL','1111':'LLLG'}for v,n in names.items():    s=I0[vec==v].abs(); idx=np.where(vec==v)[0]    print(f"{n:9s} CV={s.std()/max(s.mean(),1e-12):7.4f}   "          f"rows {idx.min():5d}-{idx.max():<5d} contiguous={(idx.max()-idx.min()+1)==len(idx)}")

## 7. Scaler ablation and figures

In [ ]:
%run src/scaling_ablation.py

In [ ]:
%run src/make_figures.py

## 8. What remains| Stage | Status ||---|---|| MATLAB validation + Coder | PENDING — run `matlab/validate_generate_c.m` || Proteus HIL | PENDING — see `proteus/proteus_settings.md`; resolve the ±900 A sensor range first || ETAP coordination | BLOCKED — network parameters absent; see `etap/etap_settings.md` |